# 02 · 单词分词、顺序切分与词表

延续旧项目的 `tokenize → build_vocab → numericalize` 流程。
关键变化是：样本来自一本连续的小说，必须**先划分正文，再建立训练词表和序列窗口**。


In [ ]:
import os
import sys
from pathlib import Path

# 支持从项目根目录或 notebooks 目录启动。
# Kaggle 上可在这里显式设置：
# os.environ["TIME_MACHINE_PROJECT_ROOT"] = "/kaggle/working/Time_Machine"
hint = os.environ.get("TIME_MACHINE_PROJECT_ROOT")
if hint:
    candidates = [Path(hint).expanduser().resolve()]
else:
    candidates = []
    for parent in (Path.cwd(), *Path.cwd().parents):
        candidates.extend([parent / "Time_Machine", parent])
PROJECT_ROOT = next(
    (path for path in candidates
     if (path / "config.py").is_file() and (path / "src/generation.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请在 Time_Machine 项目内启动，或设置 TIME_MACHINE_PROJECT_ROOT。")
existing_config = sys.modules.get("config")
if existing_config and Path(existing_config.__file__).resolve() != PROJECT_ROOT / "config.py":
    raise RuntimeError("kernel 中已经导入其他项目的 config，请重启 kernel 再运行本 Notebook。")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from config import ExperimentConfig, get_device, resolve_data_path, resolve_output_dir

data_path = resolve_data_path()
output_dir = resolve_output_dir()
device = get_device("auto")
print("项目根目录:", PROJECT_ROOT)
print("数据文件:", data_path)
print("输出目录:", output_dir)
print("计算设备:", device)


## 1. 查看分词规则

一个英文单词对应一个 ID。所有模型和推断入口都调用相同的 tokenizer。
统一小写，保留词内撇号，忽略数字和标点，连字符会把单词分开。


In [ ]:
from src.preprocessing import clean_text, tokenize, read_text, split_tokens, build_vocab, numericalize, decode, unknown_rate

example = "The Traveller’s after-dinner story: Don't stop in 1895!"
print("原文:", example)
print("清理:", clean_text(example))
print("Tokens:", tokenize(example))
body = read_text(data_path)
tokens = tokenize(body)


## 2. 按正文顺序划分 80% / 10% / 10%

推文分类可以随机切分独立样本；这本书则按 token 的原始位置连续切分。
不先创建重叠窗口再随机切分，否则相似甚至相同的文本可能落入不同数据集。
验证集用于选择最佳 epoch，测试集留作训练完成后的最终评估。


In [ ]:
config = ExperimentConfig()
splits = split_tokens(tokens, config.train_ratio, config.valid_ratio)
display(pd.DataFrame([
    {"split": name, "tokens": len(part), "first_words": " ".join(part[:12]), "last_words": " ".join(part[-12:])}
    for name, part in splits.items()
]))
print("切分后总 token 数:", sum(len(part) for part in splits.values()))
print("原始总 token 数:", len(tokens))


## 3. 只使用训练集构建词表

与旧项目一样，`<pad>=0`、`<unk>=1`，按频率降序并用单词字母顺序处理并列频率。
`min_freq=2` 表示训练集中出现至少两次才收入词表。
`<pad>` 保留以便理解，但后续固定窗口没有 padding。


In [ ]:
vocab = build_vocab([splits["train"]], min_freq=config.min_freq)
print("词表大小（含两个特殊 token）:", len(vocab))
display(pd.DataFrame(list(vocab.items())[:25], columns=["token", "id"]))
display(pd.DataFrame([
    {"split": name, "tokens": len(part), "unknown_rate": unknown_rate(part, vocab)}
    for name, part in splits.items()
]))


## 4. 从 token 到 ID，再从 ID 回到 token

未收录的词会映射成 `<unk>`，还原时无法恢复其原词。
`<unk>` 比例影响准确率和困惑度，因此后续会另外报告已知单词的准确率。


In [ ]:
example_tokens = tokenize("the time traveller qwertyunknownword")
example_ids = numericalize(example_tokens, vocab)
display(pd.DataFrame({"original_token": example_tokens, "id": example_ids, "decoded_token": decode(example_ids, vocab)}))


## 5. 观察 min_freq 的取舍

这些词表都只来自训练集。更低的阈值可以收录更多词，但其中一些词只有一个训练样本。
这里比较阈值用于理解，不用测试集选参数；正式实验从共享的 `ExperimentConfig` 读取配置。


In [ ]:
rows = []
for threshold in [1, 2, 3, 5]:
    candidate_vocab = build_vocab([splits["train"]], min_freq=threshold)
    rows.append({
        "min_freq": threshold,
        "vocab_size": len(candidate_vocab),
        "train_unknown_rate": unknown_rate(splits["train"], candidate_vocab),
        "valid_unknown_rate": unknown_rate(splits["valid"], candidate_vocab),
    })
display(pd.DataFrame(rows))


## 下一步

在 `03_dataset.ipynb` 中查看右移一位的标签、固定窗口和 batch 张量。
所有公共逻辑保存在 `src/preprocessing.py`，这里不重复定义函数。
